In [ ]:
%pip install tiktoken

In [2]:
import tiktoken
import json

encoding = tiktoken.encoding_for_model("gpt-4o") # you can change the model the tokenizer is based on here!

/Users/nithya/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
fn = "../../data/adipose_Emont2022/evidence_human/evidence.json"


## Extract Source Rationale and Target Output from Evidence Human

In [11]:
def load_json():
    """
    Reads the evidence human from global variable fn and returns a list of dictionaries storing the source rationale and associated expected target output
    """
    source_target_lst = []

    with open(fn, 'r') as file:
        evidence = json.load(file)
    
    for ev in evidence:
        source = ev['source']['source_rationale']
        
        if ev['source']['source_type'] == 'text':
            # we are comparing extracted version from evidence human to extracted version from LLM
            source_target_lst.append({'sr': source, 'target': {'ctl': ev['extracted']['cell_type_label'], 'feature': ev['extracted']['feature_name']}})

    return source_target_lst

## Tokenize Source Rationale and Target Output

In [15]:
dct = load_json()

for obj in dct:
    # tokenize the source rational and get an array of token indices in return
    sr_tokens = encoding.encode(obj['sr'])
    target_tokens = []

    # we are using extend here because we just want to ensure that the token itself is present in the source to assert that human extraction is mimicking LLM extraction properly
    target_tokens.extend(encoding.encode(obj['target']['ctl']))
    target_tokens.extend(encoding.encode(obj['target']['feature']))

    # assert that the target tokens appears exactly in sr_tokens
    if not (target_tokens[0] in sr_tokens and target_tokens[1] in sr_tokens):
        print(f"We received {obj['sr']} as input and {obj['target']['ctl']}, {obj['target']['feature']} as target output.")
        print(f"Tokenizing the source and target gave us {sr_tokens} and {target_tokens}.")